# Demo: Estoque de Emprego

Este notebook demonstra os métodos disponíveis para consulta de **estoque de emprego** da biblioteca `sdic_libraries`.

## Métodos disponíveis

| Método | Descrição |
|---|---|
| `get_estoque_emprego_nacional` | Estoque nacional com filtro por CNAE |
| `get_estoque_emprego_estadual` | Estoque por UF com filtro por CNAE |
| `get_estoque_emprego_nacional_lista_cnae` | Estoque nacional para lista específica de CNAEs (POST) |
| `get_estoque_emprego_nacional_grupos_cnae` | Estoque nacional agrupando CNAEs (POST) |
| `get_estoque_emprego_estadual_lista_cnae` | Estoque estadual para lista específica de CNAEs (POST) |
| `get_estoque_emprego_estadual_grupos_cnae` | Estoque estadual agrupando CNAEs (POST) |

## 1. Configuração

In [1]:
import sys
sys.path.insert(0, '../')

import pandas as pd
from sdic_libraries.data_access.emprego import Emprego

# Inicializar cliente
emprego = Emprego()
print('Cliente inicializado com sucesso.')

Cliente inicializado com sucesso.


.../site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


## 2. Estoque Nacional

### 2.1 Todos os setores — nível divisão (padrão)

In [2]:
dados_nacional = emprego.get_estoque_emprego_nacional(nivel_cnae=2,agregado=True)

df_nacional = pd.DataFrame(dados_nacional)

print(f'Registros retornados: {len(df_nacional)}')



colunas_nacional = [c.lower() for c in df_nacional.columns]

colunas_removidas_esperadas = [

    c for c in df_nacional.columns

    if any(token in c.lower() for token in ['uf', 'estado', 'municipio', 'grupo', 'subclasse'])

]

print('Colunas com granularidade acima do solicitado:', colunas_removidas_esperadas)



df_nacional.head()

Registros retornados: 522
Colunas com granularidade acima do solicitado: []


,ano,divisao_cnae_cod,divisao_cnae_desc,estoque_trabalhadores
0,2024,01,None,1639647
1,2024,02,None,142027
2,2024,03,None,21518
3,2024,05,None,3355
4,2024,06,None,24592


In [3]:
df_nacional.groupby('ano').agg({'estoque_trabalhadores': 'sum'}).reset_index()

,ano,estoque_trabalhadores
0,2019,46716492
1,2020,46236176
2,2021,48728871
3,2022,52790864
4,2023,55818007
5,2024,57800651


In [5]:
df_nacional.query('ano==2022').groupby('ano').agg({'estoque_trabalhadores': 'sum'}).reset_index()

,ano,estoque_trabalhadores
0,2022,52790864


### 2.2 Estoque nacional agregado (soma de todos os estados)

In [6]:
dados_nacional_agg = emprego.get_estoque_emprego_nacional(nivel_cnae=2, agregado=False)
df_nacional_agg = pd.DataFrame(dados_nacional_agg)
print(f'Registros retornados: {len(df_nacional_agg)}')
df_nacional_agg.head(50)

Registros retornados: 1000


,sigla_uf,ano,divisao_cnae_cod,divisao_cnae_desc,estoque_trabalhadores
0,AC,2024,01,None,4046
1,AC,2024,02,None,52
2,AC,2024,03,None,20
3,AC,2024,07,None,1
4,AC,2024,08,None,54
5,AC,2024,09,None,1
6,AC,2024,10,None,3934
7,AC,2024,11,None,194
8,AC,2024,13,None,11
9,AC,2024,14,None,194


### 2.3 Filtrando por CNAEs específicos — nível grupo

In [7]:
# Divisões CNAE: 10 (Fabricação de Alimentos), 26 (Equipamentos de TI), 62 (TI e Serviços)
codigos = ['01', '26', '62']

dados_nacional_cnae = emprego.get_estoque_emprego_nacional(
    codigos_cnae=codigos,
    nivel_cnae=2,
    agregado=False
)
df_nacional_cnae = pd.DataFrame(dados_nacional_cnae)
print(f'Registros retornados: {len(df_nacional_cnae)}')
df_nacional_cnae.query('sigla_uf == "SP"')

Registros retornados: 469


,sigla_uf,ano,divisao_cnae_cod,divisao_cnae_desc,estoque_trabalhadores
74,SP,2024,01,None,310850
75,SP,2024,26,None,46565
76,SP,2024,62,None,250763
152,SP,2023,01,None,319676
153,SP,2023,26,None,46466
154,SP,2023,62,None,245688
230,SP,2022,01,None,317619
231,SP,2022,26,None,48498
232,SP,2022,62,None,242729
307,SP,2021,01,None,281130


## 3. Estoque Estadual

### 3.1 Todos os setores de uma UF

In [8]:
dados_sp = emprego.get_estoque_emprego_estadual(ufs='SP', nivel_cnae=2)

df_sp = pd.DataFrame(dados_sp)

print(f'Registros retornados (SP): {len(df_sp)}')



colunas_removidas_esperadas_sp = [

    c for c in df_sp.columns

    if any(token in c.lower() for token in ['municipio', 'ibge', 'grupo', 'subclasse'])

]

print('Colunas com granularidade acima do solicitado:', colunas_removidas_esperadas_sp)



df_sp.query("ano == 2022").head()

Registros retornados (SP): 522
Colunas com granularidade acima do solicitado: []


,sigla_uf,ano,divisao_cnae_cod,divisao_cnae_desc,estoque_trabalhadores
174,SP,2022,01,None,317619
175,SP,2022,02,None,20156
176,SP,2022,03,None,1787
177,SP,2022,05,None,8
178,SP,2022,06,None,61


### 3.2 Filtrando por CNAEs para uma UF

In [9]:
dados_sp_cnae = emprego.get_estoque_emprego_estadual(

    ufs='SP',

    codigos_cnae=['10', '26', '62'],

    nivel_cnae=2

)

df_sp_cnae = pd.DataFrame(dados_sp_cnae)

print(f'Registros retornados (SP | CNAEs 10, 26, 62): {len(df_sp_cnae)}')

df_sp_cnae.head()

Registros retornados (SP | CNAEs 10, 26, 62): 18


,sigla_uf,ano,divisao_cnae_cod,divisao_cnae_desc,estoque_trabalhadores
0,SP,2024,10,None,455970
1,SP,2024,26,None,46565
2,SP,2024,62,None,250763
3,SP,2023,10,None,440964
4,SP,2023,26,None,46466


## 4. Estoque Nacional — Lista de CNAEs (POST)

Permite enviar uma lista maior de códigos via corpo da requisição.

In [10]:
lista_cnae = ['10', '13', '14', '15', '16', '17', '18', '19', '20']

dados_lista = emprego.get_estoque_emprego_nacional_lista_cnae(
    codigos_cnae=lista_cnae,
    nivel_cnae=2,
    agregado=True
)
df_lista = pd.DataFrame(dados_lista)
print(f'Registros retornados: {len(df_lista)}')
df_lista.head()

Registros retornados: 54


,ano,divisao_cnae_cod,divisao_cnae_desc,estoque_trabalhadores
0,2024,10,None,1931701
1,2024,13,None,267360
2,2024,14,None,533898
3,2024,15,None,328759
4,2024,16,None,174941


## 5. Estoque Nacional — Grupos de CNAEs (POST)

Agrupa múltiplos CNAEs sob um nome personalizado.

In [11]:
grupos = [
    {
        'nome_grupo': 'Indústria de Transformação',
        'codigos_cnae': ['10', '13', '14', '15', '16', '17', '18']
    },
    {
        'nome_grupo': 'Tecnologia da Informação',
        'codigos_cnae': ['26', '61', '62', '63']
    }
]

dados_grupos = emprego.get_estoque_emprego_nacional_grupos_cnae(
    grupos_cnae=grupos,
    nivel_cnae=2,
    agregado=True
)
df_grupos = pd.DataFrame(dados_grupos)
print(f'Registros retornados: {len(df_grupos)}')
df_grupos.head()

Registros retornados: 12


,nome_grupo,ano,estoque_trabalhadores
0,Indústria de Transformação,2024,3544109
1,Tecnologia da Informação,2024,1191642
2,Indústria de Transformação,2023,3450035
3,Tecnologia da Informação,2023,1152234
4,Indústria de Transformação,2022,3450346


## 6. Estoque Estadual — Lista de CNAEs (POST)

In [12]:
dados_rj_lista = emprego.get_estoque_emprego_estadual_lista_cnae(

    ufs='RJ',

    codigos_cnae=['10', '13', '14', '20'],

    nivel_cnae=2,


)

df_rj_lista = pd.DataFrame(dados_rj_lista)

print(f'Registros retornados (RJ): {len(df_rj_lista)}')

df_rj_lista.head()

Registros retornados (RJ): 24


,sigla_uf,ano,divisao_cnae_cod,divisao_cnae_desc,estoque_trabalhadores
0,RJ,2024,10,None,44072
1,RJ,2024,13,None,5114
2,RJ,2024,14,None,37770
3,RJ,2024,20,None,14633
4,RJ,2023,10,None,43124


## 7. Estoque Estadual — Grupos de CNAEs (POST)

In [13]:
grupos_mg = [

    {

        'nome_grupo': 'Agronegócio',

        'codigos_cnae': ['01', '02', '03', '10', '11']

    },

    {

        'nome_grupo': 'Construção Civil',

        'codigos_cnae': ['41', '42', '43']

    }

]



dados_mg_grupos = emprego.get_estoque_emprego_estadual_grupos_cnae(

    ufs='MG',

    grupos_cnae=grupos_mg,

    nivel_cnae=2

)

df_mg_grupos = pd.DataFrame(dados_mg_grupos)

print(f'Registros retornados (MG): {len(df_mg_grupos)}')

df_mg_grupos.head()

Registros retornados (MG): 12


,nome_grupo,sigla_uf,ano,estoque_trabalhadores
0,Agronegócio,MG,2024,540735
1,Construção Civil,MG,2024,361450
2,Agronegócio,MG,2023,528642
3,Construção Civil,MG,2023,357510
4,Agronegócio,MG,2022,521494


## 8. Comparativo entre estados

Exemplo de uso iterando sobre múltiplas UFs.

In [14]:
lista_ufs = ['SP', 'RJ', 'MG', 'RS', 'PR']

cnae_ti = ['62']



# A nova API aceita múltiplas UFs em uma única chamada

dados_comparativo = emprego.get_estoque_emprego_estadual(

    ufs=lista_ufs,

    codigos_cnae=cnae_ti,

    nivel_cnae=2

)

df_comparativo = pd.DataFrame(dados_comparativo)

print(f'Total de registros: {len(df_comparativo)}')

df_comparativo.head(10)

Total de registros: 30


,sigla_uf,ano,divisao_cnae_cod,divisao_cnae_desc,estoque_trabalhadores
0,MG,2024,62,None,58351
1,PR,2024,62,None,36347
2,RJ,2024,62,None,46332
3,RS,2024,62,None,43205
4,SP,2024,62,None,250763
5,MG,2023,62,None,55970
6,PR,2023,62,None,34063
7,RJ,2023,62,None,46002
8,RS,2023,62,None,39708
9,SP,2023,62,None,245688
